## 1. Serve MYH data

Go into this page in [Myndigheten för yrkeshögskola (MYH)](https://www.myh.se/yrkeshogskolan/resultat-ansokningsomgangar/resultat-for-program) and download Resultat ansökningsomgång 2024.

> [!NOTE]
> This dataset is in Swedish

We will in this exercise create an API to serve this dataset for downstream users.

a) Start with doing EDA on this dataset in a Jupyter notebook. Especially on "Tabell 3".

b) Make an API endpoint where you serve table 3 in JSON format for a read operation.

c) Make endpoints where you could filter out a particular school.

d) Make endpoints where you could filter out a particular field.

e) Make endpoint for approved (beviljad) and one for not approved (avslag).

f) Make an endpoint for some KPIs that you think is interesting for a particular stakeholder in mind.

g) What else do you want to be able to serve?

a) Start with doing EDA on this dataset in a Jupyter notebook. Especially on "Tabell 3".


In [1]:
import pandas as pd
df = pd.read_excel("resultat-ansokningsomgang-2024(1).xlsx", sheet_name="Tabell 3", header=5)
df.head(2)

,Utbildningsområde,SUN5 inriktning,SUN5 inriktning namn,Utbildningsnamn,Beslut,Diarienummer,Flera kommuner,Antal kommuner,Län,Kommun,...,Sökta utbildningsomgångar,Beviljade utbildningsomgångar,Sökta platser per utbildningsomgång,Sökta platser totalt,Beviljade platser utbildningsomgång 1,Beviljade platser utbildningsomgång 2,Beviljade platser utbildningsomgång 3,Beviljade platser utbildningsomgång 4,Beviljade platser utbildningsomgång 5,Beviljade platser totalt
0,Data/IT,481ab,"Utbildningar till programmerare, spel",AI Programmer,Avslag,MYH 2024/3742,Nej,1,Västerbotten,Umeå,...,3,0,30,90,0,0,0,0,0,0
1,Data/IT,481ab,"Utbildningar till programmerare, spel","Forsbergs, Spelutvecklare – Game Programming",Avslag,MYH 2024/4174,Nej,1,Stockholm,Stockholm,...,3,0,20,60,0,0,0,0,0,0


In [3]:
df.columns, df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1272 entries, 0 to 1271
Data columns (total 28 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Utbildningsområde                         1272 non-null   object 
 1   SUN5 inriktning                           1272 non-null   object 
 2   SUN5 inriktning namn                      1272 non-null   object 
 3   Utbildningsnamn                           1272 non-null   object 
 4   Beslut                                    1272 non-null   object 
 5   Diarienummer                              1272 non-null   object 
 6   Flera kommuner                            1272 non-null   object 
 7   Antal kommuner                            1272 non-null   int64  
 8   Län                                       1272 non-null   object 
 9   Kommun                                    1272 non-null   object 
 10  YH-poäng                            

(Index(['Utbildningsområde', 'SUN5 inriktning', 'SUN5 inriktning namn',
        'Utbildningsnamn', 'Beslut', 'Diarienummer', 'Flera kommuner',
        'Antal kommuner', 'Län', 'Kommun', 'YH-poäng', 'Studieform',
        'Studietakt %', 'Typ av examen', 'SeQF nivå', 'Smalt yrkesområde',
        'Utbildningsanordnare administrativ enhet', 'Huvudmannatyp',
        'Sökta utbildningsomgångar', 'Beviljade utbildningsomgångar',
        'Sökta platser per utbildningsomgång', 'Sökta platser totalt',
        'Beviljade platser utbildningsomgång 1',
        'Beviljade platser utbildningsomgång 2',
        'Beviljade platser utbildningsomgång 3',
        'Beviljade platser utbildningsomgång 4',
        'Beviljade platser utbildningsomgång 5', 'Beviljade platser totalt'],
       dtype='object'),
 None)

In [2]:
df_cleaned = df.drop(columns=['SUN5 inriktning','Diarienummer','Beviljade platser utbildningsomgång 1','Beviljade platser utbildningsomgång 2','Beviljade platser utbildningsomgång 3','Beviljade platser utbildningsomgång 4','Beviljade platser utbildningsomgång 5','SeQF nivå', 'Sökta platser per utbildningsomgång','Sökta utbildningsomgångar','Beviljade utbildningsomgångar'])
df_cleaned.head(2)

,Utbildningsområde,SUN5 inriktning namn,Utbildningsnamn,Beslut,Flera kommuner,Antal kommuner,Län,Kommun,YH-poäng,Studieform,Studietakt %,Typ av examen,Smalt yrkesområde,Utbildningsanordnare administrativ enhet,Huvudmannatyp,Sökta platser totalt,Beviljade platser totalt
0,Data/IT,"Utbildningar till programmerare, spel",AI Programmer,Avslag,Nej,1,Västerbotten,Umeå,550,Bunden,100,Yrkeshögskoleexamen,Nej,Futuregames Umeå,Privat,90,0
1,Data/IT,"Utbildningar till programmerare, spel","Forsbergs, Spelutvecklare – Game Programming",Avslag,Nej,1,Stockholm,Stockholm,550,Bunden,100,Yrkeshögskoleexamen,Nej,Forsbergs Skola,Privat,60,0


In [5]:
df_cleaned.describe().T

,count,mean,std,min,25%,50%,75%,max
Antal kommuner,1272.0,1.432390,0.979806,1.0,1.0,1.0,1.0,9.0
YH-poäng,1272.0,362.837264,89.905455,100.0,300.0,400.0,400.0,999.0
Studietakt %,1272.0,96.206761,12.848528,50.0,100.0,100.0,100.0,100.0
Sökta platser totalt,1272.0,106.927673,34.995530,12.0,90.0,105.0,135.0,175.0
Beviljade platser totalt,1272.0,23.080189,39.772265,0.0,0.0,0.0,48.0,175.0



b) Make an API endpoint where you serve table 3 in JSON format for a read operation.


In [6]:
df_cleaned["SUN5 inriktning namn"].nunique() #200
df_cleaned["Utbildningsnamn"].nunique() #851
df_cleaned["Utbildningsanordnare administrativ enhet"].nunique() #250 skolnamn

df_dict = df_cleaned.to_dict(orient="records")
# skolor = df_dict.get("Utbildningsanordnare administrativ enhet")
df_dict

[{'Utbildningsområde': 'Data/IT',
  'SUN5 inriktning namn': 'Utbildningar till programmerare, spel',
  'Utbildningsnamn': 'AI Programmer',
  'Beslut': 'Avslag',
  'Flera kommuner': 'Nej',
  'Antal kommuner': 1,
  'Län': 'Västerbotten',
  'Kommun': 'Umeå',
  'YH-poäng': 550,
  'Studieform': 'Bunden',
  'Studietakt %': 100,
  'Typ av examen': 'Yrkeshögskoleexamen',
  'Smalt yrkesområde': 'Nej',
  'Utbildningsanordnare administrativ enhet': 'Futuregames Umeå',
  'Huvudmannatyp': 'Privat',
  'Sökta platser totalt': 90,
  'Beviljade platser totalt': 0},
 {'Utbildningsområde': 'Data/IT',
  'SUN5 inriktning namn': 'Utbildningar till programmerare, spel',
  'Utbildningsnamn': 'Forsbergs, Spelutvecklare – Game Programming',
  'Beslut': 'Avslag',
  'Flera kommuner': 'Nej',
  'Antal kommuner': 1,
  'Län': 'Stockholm',
  'Kommun': 'Stockholm',
  'YH-poäng': 550,
  'Studieform': 'Bunden',
  'Studietakt %': 100,
  'Typ av examen': 'Yrkeshögskoleexamen',
  'Smalt yrkesområde': 'Nej',
  'Utbildningsan

In [7]:
# df_dict.get("Futuregames Umeå")

In [8]:
answer = "Futuregames Umeå"
for row in df_dict: #.get("Utbildningsanordnare administrativ enhet"): # == "0: Futuregames Umeå"
    if answer in row.get("Utbildningsanordnare administrativ enhet"):
        print(row)
    # print(row)
# df_dict.get("Utbildningsanordnare administrativ enhet")

{'Utbildningsområde': 'Data/IT', 'SUN5 inriktning namn': 'Utbildningar till programmerare, spel', 'Utbildningsnamn': 'AI Programmer', 'Beslut': 'Avslag', 'Flera kommuner': 'Nej', 'Antal kommuner': 1, 'Län': 'Västerbotten', 'Kommun': 'Umeå', 'YH-poäng': 550, 'Studieform': 'Bunden', 'Studietakt %': 100, 'Typ av examen': 'Yrkeshögskoleexamen', 'Smalt yrkesområde': 'Nej', 'Utbildningsanordnare administrativ enhet': 'Futuregames Umeå', 'Huvudmannatyp': 'Privat', 'Sökta platser totalt': 90, 'Beviljade platser totalt': 0}
{'Utbildningsområde': 'Data/IT', 'SUN5 inriktning namn': 'Utbildningar till projektledare IT', 'Utbildningsnamn': 'Futuregames Project Manager IT & Games', 'Beslut': 'Avslag', 'Flera kommuner': 'Ja', 'Antal kommuner': 3, 'Län': 'Flera kommuner', 'Kommun': 'Flera kommuner', 'YH-poäng': 300, 'Studieform': 'Distans', 'Studietakt %': 100, 'Typ av examen': 'Yrkeshögskoleexamen', 'Smalt yrkesområde': 'Nej', 'Utbildningsanordnare administrativ enhet': 'Futuregames Umeå', 'Huvudmann

In [7]:
len(df_cleaned[df_cleaned["Beslut"] == "Beviljad"])
df_cleaned.columns

Index(['Utbildningsområde', 'SUN5 inriktning namn', 'Utbildningsnamn',
       'Beslut', 'Flera kommuner', 'Antal kommuner', 'Län', 'Kommun',
       'YH-poäng', 'Studieform', 'Studietakt %', 'Typ av examen',
       'Smalt yrkesområde', 'Utbildningsanordnare administrativ enhet',
       'Huvudmannatyp', 'Sökta platser totalt', 'Beviljade platser totalt'],
      dtype='object')

200


c) Make endpoints where you could filter out a particular school.



d) Make endpoints where you could filter out a particular field.


In [10]:
df_cleaned.head(1) # Field = Utbildningsområde? 

,Utbildningsområde,SUN5 inriktning namn,Utbildningsnamn,Beslut,Flera kommuner,Antal kommuner,Län,Kommun,YH-poäng,Studieform,Studietakt %,Typ av examen,Smalt yrkesområde,Utbildningsanordnare administrativ enhet,Huvudmannatyp,Sökta platser totalt,Beviljade platser totalt
0,Data/IT,"Utbildningar till programmerare, spel",AI Programmer,Avslag,Nej,1,Västerbotten,Umeå,550,Bunden,100,Yrkeshögskoleexamen,Nej,Futuregames Umeå,Privat,90,0



e) Make endpoint for approved (beviljad) and one for not approved (avslag).


In [21]:
lists = []
for row in df_dict:
    if row.get("Beslut") == "Avslag":
        lists.append(row)
# lists

len([row for row in df_dict if row.get("Beslut") == "Avslag"])

928


f) Make an endpoint for some KPIs that you think is interesting for a particular stakeholder in mind.



g) What else do you want to be able to serve?